In [1]:
import sys
print(sys.executable)
print(sys.version)

c:\Users\Fausto\AppData\Local\Python\pythoncore-3.14-64\python.exe
3.14.3 (tags/v3.14.3:323c59a, Feb  3 2026, 16:04:56) [MSC v.1944 64 bit (AMD64)]


In [2]:
import duckdb
print("duckdb ok")

duckdb ok


In [5]:
import duckdb

con = duckdb.connect()

path = r"C:\Users\Fausto\Downloads\train_rsample_100k.parquet"

df = con.execute(f"""
    SELECT *
    FROM read_parquet('{path}')
""").df()

print(df.shape)
print(df.head())

(100000, 10)
   datetime ui_language selected_template              eligible_templates  \
0  9.283380          fr                 L     [K, H, G, E, B, J, L, F, D]   
1  9.623044          es                 D     [K, H, G, E, B, J, L, F, D]   
2  5.172986          de                 K  [G, E, B, A, K, H, J, L, F, D]   
3  6.269595          en                 G     [G, E, B, K, H, J, L, F, D]   
4  0.913009          es                 L     [G, E, B, K, H, J, L, F, D]   

                                             history  history_length  \
0  [{'template': 'J', 'n_days': 5.882936954498291...               3   
1  [{'template': 'A', 'n_days': 23.98947525024414...              11   
2  [{'template': 'E', 'n_days': 4.508449554443359...               3   
3  [{'template': 'K', 'n_days': 8.594528198242188...               7   
4  [{'template': 'L', 'n_days': 0.9999991059303284}]               1   

   n_eligible  time_of_day  session_end_completed   hour_utc  
0           9     9.283380  

1) How big is this dataset?

In [6]:
con.execute(f"""
SELECT COUNT(*) AS n_rows
FROM read_parquet('{path}')
""").df()

,n_rows
0,100000


2) What is the global sucess rate (reward)?

In [7]:
con.execute(f"""
SELECT AVG(CASE WHEN session_end_completed THEN 1 ELSE 0 END) AS reward_rate
FROM read_parquet('{path}')
""").df()

,reward_rate
0,0.14704


In [8]:
con.execute(f"""
SELECT
  regexp_extract(filename, 'train-part-\\d+', 0) AS part,
  COUNT(*) AS n,
  AVG(CAST(session_end_completed AS INT)) AS reward_rate
FROM read_parquet('{path}', filename=true)
GROUP BY 1
ORDER BY 1
""").df()

,part,n,reward_rate
0,,100000,0.14704


3) How often occurs every template + how good does it work?

In [9]:
con.execute(f"""
SELECT
  selected_template,
  COUNT(*) AS n,
  AVG(CASE WHEN session_end_completed THEN 1 ELSE 0 END) AS reward_rate
FROM read_parquet('{path}')
GROUP BY selected_template
ORDER BY reward_rate DESC
""").df()

,selected_template,n,reward_rate
0,C,3102,0.413282
1,A,4053,0.272144
2,K,10395,0.136893
3,L,10267,0.134606
4,D,10303,0.134039
5,G,10332,0.133662
6,H,10239,0.133411
7,E,10422,0.131549
8,B,10342,0.131213
9,F,10309,0.129887


4) Does it differ per language?

In [10]:
con.execute(f"""
SELECT
  ui_language,
  COUNT(*) AS n,
  AVG(CASE WHEN session_end_completed THEN 1 ELSE 0 END) AS reward_rate
FROM read_parquet('{path}')
GROUP BY ui_language
ORDER BY n DESC
""").df()

,ui_language,n,reward_rate
0,en,40809,0.165650
1,es,24250,0.124866
2,pt,9502,0.128184
3,ru,4512,0.143839
4,fr,3859,0.156776
5,de,2871,0.199930
6,zs,1866,0.094855
7,ar,1825,0.058630
8,it,1753,0.143183
9,vi,1573,0.140496


5) Important for bandit: how many templates are eligible per event?

In [11]:
con.execute(f"""
SELECT
  AVG(LEN(eligible_templates)) AS avg_eligible,
  MIN(LEN(eligible_templates)) AS min_eligible,
  MAX(LEN(eligible_templates)) AS max_eligible
FROM read_parquet('{path}')
""").df()

,avg_eligible,min_eligible,max_eligible
0,9.14069,1,10


How often is C available?

In [12]:
con.execute(f"""
SELECT
  COUNT(*) AS total,
  SUM(CASE WHEN 'C' IN eligible_templates THEN 1 ELSE 0 END) AS c_available
FROM read_parquet('{path}')
""").df()

,total,c_available
0,100000,3102.0


What if we always choose C when C is available, 
and otherwise the best of the rest?

So firstly we must know:
What is the best template without C?

In [14]:
con.execute(f"""
SELECT
  selected_template,
  COUNT(*) AS n,
  AVG(CASE WHEN session_end_completed THEN 1 ELSE 0 END) AS reward_rate
FROM read_parquet('{path}')
WHERE 'C' NOT IN eligible_templates
GROUP BY selected_template
ORDER BY reward_rate DESC
""").df()

,selected_template,n,reward_rate
0,A,4053,0.272144
1,K,10395,0.136893
2,L,10267,0.134606
3,D,10303,0.134039
4,G,10332,0.133662
5,H,10239,0.133411
6,E,10422,0.131549
7,B,10342,0.131213
8,F,10309,0.129887
9,J,10236,0.128859


There is a clear hierarchy:
C-> best overall(41.2%)
A -> best if C is not available (26.9%)
Rest -> all around 13%

This is no subtle difference.
A strong rule-based policy would be:
If C available -> choose C
Else -> choose A

1) How often would our new policy choose C vs A?

In [15]:
con.execute(f"""
SELECT
  COUNT(*) AS total,
  SUM(CASE WHEN 'C' IN eligible_templates THEN 1 ELSE 0 END) AS policy_choose_C,
  SUM(CASE WHEN 'C' NOT IN eligible_templates AND 'A' IN eligible_templates THEN 1 ELSE 0 END) AS policy_choose_A,
  SUM(CASE WHEN 'C' NOT IN eligible_templates AND 'A' NOT IN eligible_templates THEN 1 ELSE 0 END) AS policy_choose_neither
FROM read_parquet('{path}')
""").df()

,total,policy_choose_C,policy_choose_A,policy_choose_neither
0,100000,3102.0,40530.0,56368.0


2) Off-policy estimate: reward of policy “C else A”

In [16]:
con.execute(f"""
WITH data AS (
  SELECT
    session_end_completed,
    selected_template,
    CASE 
      WHEN 'C' IN eligible_templates THEN 'C'
      ELSE 'A'
    END AS policy_template
  FROM read_parquet('{path}')
)
SELECT
  COUNT(*) AS total_events,
  SUM(CASE WHEN selected_template = policy_template THEN 1 ELSE 0 END) AS matched_events,
  AVG(CASE WHEN selected_template = policy_template THEN CASE WHEN session_end_completed THEN 1 ELSE 0 END END) AS estimated_policy_reward
FROM data
""").df()

,total_events,matched_events,estimated_policy_reward
0,100000,7155.0,0.333333


What do we see?
Total events
87,665,839
Matched events
5,996,554 (useful for evaluation)
Estimated policy reward
0.329 (32.9%)
Comparison with current logging policy(=In the historical data, Duolingo selected a template at random from the eligible pool, with equal probability for each template. That random decision process is what generated the dataset — that’s the logging policy)
Logging policy reward was:
14.4%
Our simple new rule:
If C available -> C
Else -> A
32.9%

A matched event is:
An event where the logging policy chose by chance exactly the same template as your new policy would have chosen.
Why do we only use matched events?

!!!Because we only know the reward of what was actually shown!!!

For example:
Suppose:
Eligible = {C, D, E}
Logging chose D
Our new policy would have chosen C
Then we know:
Reward of D -> yes
Reward of C -> no (counterfactual unknown)
So we can't use that event to evaluate C.

This compares C vs A in exactly the same context/set/pool
It removes a big part of the selection bias.

In [17]:
con.execute(f"""
SELECT
  selected_template,
  COUNT(*) AS n,
  AVG(CASE WHEN session_end_completed THEN 1 ELSE 0 END) AS reward_rate
FROM read_parquet('{path}')
WHERE 'C' IN eligible_templates
  AND 'A' IN eligible_templates
GROUP BY selected_template
ORDER BY reward_rate DESC
""").df()

,selected_template,n,reward_rate


What does this mean in terms of content?

!!!C and A apparently never occur in the same eligible pool!!!

So:
!!!C is shown in a completely different segment than A,
this confirms selection bias!!!
You can't compare C and A within the same context.

The rule:
If C available -> C
Else -> A
actually works because:
C-segment = high engagement users
A-segment = other users
So your policy is actually:
If user in C-segment -> choose C
Otherwise -> choose A
But that segment difference is already incorporated in the data.

1) Which eligible pools occur the most?

In [18]:
con.execute(f"""
SELECT
  eligible_templates,
  COUNT(*) AS n
FROM read_parquet('{path}')
GROUP BY eligible_templates
ORDER BY n DESC
LIMIT 20
""").df()

,eligible_templates,n
0,"[G, E, B, K, H, J, L, F, D]",40698
1,"[G, E, B, A, K, H, J, L, F, D]",31202
2,"[K, H, G, E, B, J, L, F, D]",15611
3,"[K, H, G, E, B, J, L, F, D, A]",9152
4,[C],3102
5,"[A, K, H]",129
6,"[K, H]",59
7,"[K, H, A]",47


2) Within every pool: reward per selected_template (and top/best template per pool)

In [19]:
con.execute(f"""
WITH stats AS (
  SELECT
    eligible_templates,
    selected_template,
    COUNT(*) AS n,
    AVG(CASE WHEN session_end_completed THEN 1 ELSE 0 END) AS reward_rate
  FROM read_parquet('{path}')
  GROUP BY eligible_templates, selected_template
),
ranked AS (
  SELECT
    *,
    ROW_NUMBER() OVER (
      PARTITION BY eligible_templates
      ORDER BY reward_rate DESC
    ) AS rnk
  FROM stats
  WHERE n >= 5000
)
SELECT
  eligible_templates,
  selected_template AS best_template,
  n,
  reward_rate
FROM ranked
WHERE rnk = 1
ORDER BY n DESC
LIMIT 50
""").df()

,eligible_templates,best_template,n,reward_rate


What do we see here?
There are a few big eligible pools (million events).
The best template is not always C within these pools.
In some pools wins L.
In little pools (such as [A, K, H]) wins A with a high reward rate.
C is actually in its own pool= [C].
That means:
Templates live in different worlds.
Reward rate is strongly dependent of the eligible pool.

Logistic Model:

STEP1 : load the data (fast + safe) with DuckDB in Python

In [29]:
import duckdb
import pandas as pd

# 1) Connect
con = duckdb.connect()

# 2) Zet hier je pad naar de parquet files (training of test)
# Voorbeeld: "data/training/*.parquet" of "/path/to/training/*.parquet"
PARQUET_GLOB = r"C:\Users\Fausto\Downloads\train_rsample_100k.parquet"

# 3) Maak een view (handig om later queries te hergebruiken)
con.execute(f"""
CREATE OR REPLACE VIEW notif AS
SELECT *
FROM read_parquet('{PARQUET_GLOB}');
""")

# 4) Snelle sanity checks
print("Row count (approx query):")
print(con.execute("SELECT COUNT(*) AS n FROM notif").fetchdf())

print("\nColumns + types:")
print(con.execute("DESCRIBE notif").fetchdf())

print("\nOverall reward rate:")
print(con.execute("""
SELECT AVG(CAST(session_end_completed AS INTEGER)) AS reward_rate
FROM notif
""").fetchdf())

print("\nExample rows:")
print(con.execute("""
SELECT datetime, ui_language, eligible_templates, history, selected_template, session_end_completed
FROM notif
LIMIT 3
""").fetchdf())

Row count (approx query):
        n
0  100000

Columns + types:
             column_name                                 column_type null  \
0               datetime                                      DOUBLE  YES   
1            ui_language                                     VARCHAR  YES   
2      selected_template                                     VARCHAR  YES   
3     eligible_templates                                   VARCHAR[]  YES   
4                history  STRUCT("template" VARCHAR, n_days FLOAT)[]  YES   
5         history_length                                    UINTEGER  YES   
6             n_eligible                                    UINTEGER  YES   
7            time_of_day                                      DOUBLE  YES   
8  session_end_completed                                     BOOLEAN  YES   
9               hour_utc                                      DOUBLE  YES   

    key default extra  
0  None    None  None  
1  None    None  None  
2  None    None 

Step 2: drawing a training sample + prepare features
We want:
Only real decision points (len(eligible_templates) >= 2)
A sample of ±1 miljoen rows (for speed)

In [28]:
query = """
SELECT
    ui_language,
    selected_template,
    array_length(eligible_templates) AS n_eligible,
    CAST(session_end_completed AS INTEGER) AS reward,
    (
      SELECT min(h.n_days)
      FROM unnest(history) AS u(h)
      WHERE h.template = selected_template
    ) AS days_since_last_notification,
    history_length,
    time_of_day
FROM notif
WHERE array_length(eligible_templates) >= 2
USING SAMPLE 1000000 ROWS
"""

df = con.execute(query).df()

print(df.shape)
print(df.head())
print("Reward rate in sample:", df['reward'].mean())

(96898, 7)
  ui_language selected_template  n_eligible  reward  \
0          en                 F           9       0   
1          ru                 A          10       0   
2          es                 J           9       0   
3          es                 J          10       0   
4          es                 H           9       0   

   days_since_last_notification  history_length  time_of_day  
0                           NaN               5     5.254317  
1                           NaN               0     5.501725  
2                      1.000006               1     4.285660  
3                     16.101513              15     8.757106  
4                           NaN              15     3.556493  
Reward rate in sample: 0.13851679085223637


Step 3 — Logistic Regression training

In [24]:
!pip install scikit-learn

'pip' is not recognized as an internal or external command,
operable program or batch file.


In [30]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score

x = df[['selected_template', 'ui_language','history_length', 'n_eligible', 'time_of_day', 'days_since_last_notification']]
y = df['reward']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

preprocess = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'),
         ['ui_language', 'selected_template']),
        ('num', 'passthrough', ['n_eligible'])
    ]
)

model = Pipeline([
    ('preprocess', preprocess),
    ('clf', LogisticRegression(max_iter=1000))
])

model.fit(X_train, y_train)

y_pred = model.predict_proba(X_test)[:, 1]
auc = roc_auc_score(y_test, y_pred)

print("AUC:", auc)

AUC: 0.4926070183452158


In [33]:
# =========================
# Step 4A — New sample WITH extra features
# =========================
query = """
SELECT
    ui_language,
    selected_template,
    eligible_templates,
    array_length(eligible_templates) AS n_eligible,
    CAST(session_end_completed AS INTEGER) AS reward,

    -- extra features you used in training
    history_length,
    time_of_day,

    (
        SELECT min(h.n_days)
        FROM unnest(history) AS u(h)
        WHERE h.template = selected_template
    ) AS days_since_last_notification

FROM notif
WHERE array_length(eligible_templates) >= 2
USING SAMPLE 300000 ROWS
"""

df_policy = con.execute(query).df()
print(df_policy.shape)
print(df_policy.head())


# =========================
# Step 4B — Policy simulation (greedy) WITH extra features
# =========================
import numpy as np
import pandas as pd

df_policy = df_policy.reset_index(drop=True)
df_policy["row_id"] = np.arange(len(df_policy))

# explode eligible templates -> one row per candidate template
long = df_policy[
    ["row_id", "ui_language", "n_eligible", "reward",
     "selected_template", "eligible_templates",
     "history_length", "time_of_day", "days_since_last_notification"]
].explode("eligible_templates")

long = long.rename(columns={"eligible_templates": "cand_template"})

# Build X_long with the SAME columns as training X
X_long = long[
    ["ui_language", "n_eligible", "history_length", "time_of_day", "days_since_last_notification"]
].copy()
X_long["selected_template"] = long["cand_template"].astype(str)

# If days_since_last_notification has missing values, decide a fill (same idea as before)
# Option 1: fill with a large number (= "not seen recently")
X_long["days_since_last_notification"] = X_long["days_since_last_notification"].fillna(999)

# predict probabilities for each candidate template
long["p_hat"] = model.predict_proba(X_long)[:, 1]

# choose best template per original row
best_idx = long.groupby("row_id")["p_hat"].idxmax()
chosen = long.loc[best_idx, ["row_id", "cand_template"]].rename(columns={"cand_template": "model_choice"})

# join back and evaluate matches
df_eval = df_policy.merge(chosen, on="row_id", how="left")

matched = df_eval[df_eval["model_choice"] == df_eval["selected_template"]]

matched_fraction = len(matched) / len(df_eval)
reward_matched = matched["reward"].mean()
reward_baseline = df_eval["reward"].mean()

print("Matched fraction:", matched_fraction)
print("Reward in matched cases:", reward_matched)
print("Random baseline reward:", reward_baseline)
print("Absolute uplift:", reward_matched - reward_baseline)
print("Relative uplift:", (reward_matched / reward_baseline) - 1)

(96898, 8)
  ui_language selected_template              eligible_templates  n_eligible  \
0          en                 E     [G, E, B, K, H, J, L, F, D]           9   
1          es                 B  [G, E, B, A, K, H, J, L, F, D]          10   
2          en                 K  [G, E, B, A, K, H, J, L, F, D]          10   
3          es                 F     [G, E, B, K, H, J, L, F, D]           9   
4          en                 D  [K, H, G, E, B, J, L, F, D, A]          10   

   reward  history_length  time_of_day  days_since_last_notification  
0       0               7     6.895671                      4.840139  
1       0              19     5.799120                     10.925735  
2       0              16     2.608218                      9.620344  
3       0               2     7.420093                           NaN  
4       0              13    10.411852                           NaN  
Matched fraction: 0.10636828776454031
Reward in matched cases: 0.13851679085223637
Rando

Wat does 0.49 AUC mean concrete?
0.5 = random guessing
0.6 = poor signal
0.7 = solidly predictive
0.8+ = strong model
So:
Template doesn't have causal impact.
Your model can have AUC, but:
maybe predicts especially user/context effect.
and not template effect.
So:
Model says: "This is a good user-moment"
But whatever template you show doesn't really matter.
Then:
Every policy ≈ random policy
En IPS zal exact baseline geven.


Stap 4C — Check template ranking according to model

In [31]:
import numpy as np

# Extract coefficients
feature_names = model.named_steps['preprocess'].get_feature_names_out()
coefs = model.named_steps['clf'].coef_[0]

coef_df = pd.DataFrame({
    "feature": feature_names,
    "coef": coefs
})

# Kijk enkel naar template effecten
template_effects = coef_df[coef_df["feature"].str.contains("selected_template")]

template_effects.sort_values("coef", ascending=False).head(10)

,feature,coef
32,cat__selected_template_L,-0.101611
24,cat__selected_template_B,-0.111611
25,cat__selected_template_D,-0.123440
30,cat__selected_template_J,-0.146628
26,cat__selected_template_E,-0.152655
29,cat__selected_template_H,-0.157053
27,cat__selected_template_F,-0.160102
28,cat__selected_template_G,-0.167169
31,cat__selected_template_K,-0.172737
23,cat__selected_template_A,-0.215299


In [36]:
import numpy as np
import pandas as pd

# =========================
# STEP 4B — Policy simulation (recency-aware features included)
# =========================

# df_policy must contain at least:
# ['ui_language','selected_template','eligible_templates','n_eligible','reward',
#  'history_length','time_of_day','days_since_last_notification',
#  'days_since_shown' (optional if you computed it)]
#
# NOTE: If you do NOT have some columns in df_policy, remove them from feature_cols below.

df_policy = df_policy.reset_index(drop=True)
df_policy["row_id"] = np.arange(len(df_policy))

# 1) explode eligible_templates -> one row per candidate template
long = df_policy[
    ["row_id", "eligible_templates", "selected_template", "reward",
     "ui_language", "n_eligible",
     "history_length", "time_of_day", "days_since_last_notification"]
].explode("eligible_templates")

long = long.rename(columns={"eligible_templates": "cand_template"})

# 2) build X_long with ALL features your model expects
#    We set selected_template = candidate template, because the model learns p(reward | template, context)
feature_cols = [
    "selected_template",
    "ui_language",
    "history_length",
    "n_eligible",
    "time_of_day",
    "days_since_last_notification",
]

X_long = pd.DataFrame({
    "selected_template": long["cand_template"].astype(str),
    "ui_language": long["ui_language"].astype(str),
    "history_length": long["history_length"],
    "n_eligible": long["n_eligible"],
    "time_of_day": long["time_of_day"],
    "days_since_last_notification": long["days_since_last_notification"],
})

# If you added other engineered columns (e.g. days_since_shown), include them here too:
# Example:
# if "days_since_shown" in df_policy.columns:
#     feature_cols.append("days_since_shown")
#     X_long["days_since_shown"] = long["days_since_shown"]

# 3) predict p_hat for each candidate
long["p_hat"] = model.predict_proba(X_long)[:, 1]

# 4) choose best candidate per row_id (greedy policy)
best_idx = long.groupby("row_id")["p_hat"].idxmax()
chosen = long.loc[best_idx, ["row_id", "cand_template"]].rename(columns={"cand_template": "model_choice"})

# 5) join back to original rows
df_eval = df_policy.merge(chosen, on="row_id", how="left")

# =========================
# STEP 4C — Matched-only sanity (NOT an estimator)
# =========================
matched = df_eval[df_eval["model_choice"] == df_eval["selected_template"]]

matched_fraction = len(matched) / len(df_eval)
reward_matched = matched["reward"].mean()
reward_baseline = df_eval["reward"].mean()

print("Matched fraction:", matched_fraction)
print("Reward in matched cases:", reward_matched)
print("Baseline reward:", reward_baseline)
print("Matched uplift (NOT causal):", reward_matched - reward_baseline)

# =========================
# STEP 5 — IPS evaluation (causal estimate under uniform random logging)
# =========================
# Assumption: logging policy chose uniformly from eligible_templates
# => b(a|x) = 1/n_eligible
# IPS weight = 1/b = n_eligible

df_eval["match"] = (df_eval["model_choice"] == df_eval["selected_template"]).astype(int)
df_eval["w"] = df_eval["n_eligible"].astype(float)

ips_estimate = (df_eval["match"] * df_eval["reward"] * df_eval["w"]).mean()
baseline = df_eval["reward"].mean()

print("\nIPS estimated reward of model-policy:", ips_estimate)
print("Random baseline reward:", baseline)
print("Absolute uplift:", ips_estimate - baseline)
print("Relative uplift:", ips_estimate / baseline - 1)

# Optional: quick IPS sanity checks
print("\nSanity checks:")
print("E[match] =", df_eval["match"].mean(), "   vs   E[1/n_eligible] =", (1/df_eval["n_eligible"]).mean())
print("E[match*w] =", (df_eval["match"] * df_eval["w"]).mean(), " (should be ~1 if uniform logging)")

Matched fraction: 0.10636828776454031
Reward in matched cases: 0.13851679085223637
Baseline reward: 0.14392178860485616
Matched uplift (NOT causal): -0.0054049977526197945

IPS estimated reward of model-policy: 0.14392178860485616
Random baseline reward: 0.14392178860485616
Absolute uplift: 0.0
Relative uplift: 0.0

Sanity checks:
E[match] = 0.10636828776454031    vs   E[1/n_eligible] = 0.10636828776454035
E[match*w] = 1.0  (should be ~1 if uniform logging)


Is it necessary to test the model further if AUC = 0.49?

Short answer: No.

An AUC of 0.49 means the model performs essentially at random level (0.5). Testing it further on another test set will not change the fundamental conclusion: